In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report,confusion_matrix

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/QoS Dataset/network_dataset_labeled.csv')

In [ ]:
df.columns

Index(['timestamp', 'bandwidth', 'throughput', 'congestion', 'packet_loss',
       'latency', 'jitter', 'Routers', 'Planned route', 'Network measure',
       'Network target', 'Video target', 'Percentage video occupancy',
       'Bitrate video', 'Number videos', 'anomaly_throughput',
       'anomaly_congestion', 'anomaly_packet_loss', 'anomaly_latency',
       'anomaly_jitter', 'anomaly'],
      dtype='object')

In [ ]:
anomaly_cols = [col for col in df.columns if col.startswith('anomaly_') and col != 'anomaly']

anomaly_1_df = df[df['anomaly'] == 1]

if not anomaly_1_df.empty:
    print(f"When 'anomaly' is 1, here's how often other anomaly columns are also 1:")
    for col in anomaly_cols:
        # Summing because 1 indicates an anomaly
        count_both_1 = anomaly_1_df[col].sum()
        total_anomaly_1 = len(anomaly_1_df)
        if total_anomaly_1 > 0:
            percentage = (count_both_1 / total_anomaly_1) * 100
            print(f"- '{col}' is 1 in {int(count_both_1)} out of {total_anomaly_1} cases ({percentage:.2f}%)")
        else:
            print(f"- No cases where 'anomaly' is 1.")
elif 'anomaly' in df.columns:
    print("There are no rows where the 'anomaly' column is set to 1.")
else:
    print("The 'anomaly' column does not exist in the DataFrame.")

When 'anomaly' is 1, here's how often other anomaly columns are also 1:
- 'anomaly_throughput' is 1 in 26 out of 83 cases (31.33%)
- 'anomaly_congestion' is 1 in 42 out of 83 cases (50.60%)
- 'anomaly_packet_loss' is 1 in 60 out of 83 cases (72.29%)
- 'anomaly_latency' is 1 in 25 out of 83 cases (30.12%)
- 'anomaly_jitter' is 1 in 27 out of 83 cases (32.53%)


In [ ]:
anomaly_1_df['num_related_anomalies'] = anomaly_1_df[anomaly_cols].sum(axis=1)

print("Distribution of how many other 'anomaly_' columns are 1 when 'anomaly' is 1:")
print(anomaly_1_df['num_related_anomalies'].value_counts().sort_index())

Distribution of how many other 'anomaly_' columns are 1 when 'anomaly' is 1:
num_related_anomalies
2    69
3    14
Name: count, dtype: int64


/tmp/ipykernel_275/2884079021.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  anomaly_1_df['num_related_anomalies'] = anomaly_1_df[anomaly_cols].sum(axis=1)


In [ ]:
df['num_anomalies'] = df[anomaly_cols].sum(axis=1)
display(df.head())

,timestamp,bandwidth,throughput,congestion,packet_loss,latency,jitter,Routers,Planned route,Network measure,...,Percentage video occupancy,Bitrate video,Number videos,anomaly_throughput,anomaly_congestion,anomaly_packet_loss,anomaly_latency,anomaly_jitter,anomaly,num_anomalies
0,2024-05-11 12:00:15,2,2.15,0.38,0.0,6.58,0.52,up xrv6,Best effort,S1,...,0,0,0,0,0,0,0,0,0,0
1,2024-05-11 12:00:43,2,2.16,0.12,0.0,5.36,0.34,up xrv6,Best effort,S1,...,0,0,0,0,0,0,0,0,0,0
2,2024-05-11 12:01:12,2,2.00,0.08,0.0,6.29,0.23,up xrv6,Best effort,S1,...,0,0,0,0,0,0,0,0,0,0
3,2024-05-11 12:01:40,2,2.07,0.07,0.0,5.91,0.51,up xrv6,Best effort,S1,...,0,0,0,0,0,0,0,0,0,0
4,2024-05-11 12:02:08,2,2.40,0.08,0.0,5.81,0.71,up xrv6,Best effort,S1,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df.num_anomalies.value_counts()

,count
num_anomalies,
0,747
1,171
2,69
3,14


On inspection if two or more num_anomlies are present then the dataset is marking the packet/flow as an anomaly

In [ ]:
features = ["throughput", "latency", "jitter", "packet_loss","congestion"]

df = df[features + ["anomaly"]]  # keep label for evaluation

df = df.dropna()

print(df.head())

   throughput  latency  jitter  packet_loss  congestion  anomaly
0        2.15     6.58    0.52          0.0        0.38        0
1        2.16     5.36    0.34          0.0        0.12        0
2        2.00     6.29    0.23          0.0        0.08        0
3        2.07     5.91    0.51          0.0        0.07        0
4        2.40     5.81    0.71          0.0        0.08        0


In [ ]:
X = df[["throughput", "latency", "jitter", "packet_loss", "congestion"]]
y = df["anomaly"]

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
model = IsolationForest(
    n_estimators=100,
    contamination=0.1,
    random_state=42
)

model.fit(X_scaled)

IsolationForest(contamination=0.1, random_state=42)

In [ ]:
preds = model.predict(X_scaled)

preds = np.where(preds == -1, 1, 0)

In [ ]:
print(confusion_matrix(y, preds))
print(classification_report(y, preds))

[[883  35]
 [ 18  65]]
              precision    recall  f1-score   support

           0       0.98      0.96      0.97       918
           1       0.65      0.78      0.71        83

    accuracy                           0.95      1001
   macro avg       0.82      0.87      0.84      1001
weighted avg       0.95      0.95      0.95      1001



Got decent results on the dataset but need to check out on real data

Basically the model is predicting more anamolies than they are present in the dataset